# T1. 서울 실시간 도시 상황판

서울 실시간 도시데이터와 따릉이 실시간 API 를 Kafka 로 흘려 MinIO 레이크에 쌓고, Spark 로 혼잡도 표를 만든 뒤, 같은 응답 속 행사 · 통제 · 날씨 글을 Qdrant 에 넣어 「지금 광화문 왜 붐벼?」에 근거를 붙여 답합니다. 매시간 도는 Airflow DAG 까지 만듭니다.

- 단계: 팀 프로젝트 · 정형 + 비정형 · 예상 3일(팀 규모 · 혼자 진행)
- 실행 환경: 내 컴퓨터의 Docker + Jupyter(VS Code 노트북도 됩니다)
- 데이터: 서울 열린데이터광장 — 서울시 실시간 도시데이터(OA-21285, OpenAPI citydata), 공공누리 1유형(출처표시)
- 데이터: 서울 열린데이터광장 — 서울시 공공자전거 따릉이 실시간 대여정보(OA-15493, OpenAPI bikeList), 공공누리 1유형(출처표시)

셀을 위에서부터 차례로 실행하세요. 각 단계 끝의 **확인 포인트** 숫자가 같게 나오면 다음 단계로 갑니다.
따라하기를 한 번 끝냈다면 같은 내용의 [빈칸 노트북](https://colab.research.google.com/github/aebonlee/materials/blob/main/data-eng/projects/T1/T1_blank.ipynb)으로 다시 쳐 보세요.

서울시는 주요 장소 100여 곳의 「지금 상황」을 5분마다 갱신해 OpenAPI 로 공개합니다(실시간 도시데이터).
응답 한 건에 인구 혼잡도 같은 **숫자**와 문화행사 · 공사 통제 · 날씨 안내 같은 **글**이 함께 들어 있습니다.
이 프로젝트는 그 한 출처로 정형과 비정형을 모두 다룹니다 — 숫자는 Spark 로 집계하고, 글은 RAG 의 근거 문서로 씁니다.

```text
citydata · bikeList API ─▶ Producer ─▶ Kafka(city.live · bike.live) ─▶ Consumer ─▶ MinIO(lake/raw)
                                                                                    │
                                              Spark 혼잡도 집계 ─▶ MinIO(lake/mart) ◀┘
                                              글 → 임베딩 ─▶ Qdrant ─▶ 「지금 광화문 왜 붐벼?」
Airflow DAG(매시간) 가 위 네 일을 차례로 부릅니다.
```

팀 하나가 할 만한 규모라 일을 네 갈래로 나눕니다. 단계 제목 옆의 갈래가 그 단계에서 맡는 일입니다. 혼자서 네 갈래를 차례로 맡아 노트북 하나로 끝까지 돌려 보고, 그다음 갈래마다 하나씩 키우세요.

| 역할 | 맡는 단계 |
|---|---|
| 수집 · 스트리밍 | 3 API 수집 · 4 Kafka Producer · 5 Consumer → MinIO |
| 레이크 · Spark | 2 인프라 · 6 Spark 혼잡도 |
| RAG · 서비스 | 7 문서 · Qdrant · 8 질문에 답하기 |
| 오케스트레이션 | 9 Airflow DAG · 10 정리 |

다 끝내면 다음을 말할 수 있습니다.

- 실시간 API 응답이 Kafka 토픽을 거쳐 레이크의 어느 경로에 어떤 파일로 쌓이는지
- 같은 시각 자료를 두 번 받았을 때 Spark 에서 어떻게 한 번만 세는지
- 「왜 붐벼?」라는 질문에 숫자(혼잡도 표)와 글(행사 · 통제 문서)을 어떻게 함께 붙이는지

> **자주 겪는 오류** — 이 노트북은 **내 컴퓨터의 Docker** 에서 돕니다(Colab 에서는 Kafka · MinIO 를 띄울 수 없습니다). Docker Desktop 을 켜 두고, 메모리는 6GB 이상 주세요.
> 실시간 데이터라 실행할 때마다 숫자가 달라집니다. 그래서 확인 포인트는 숫자 대신 **형태**(열 이름 · 행 수가 1 이상 · 값의 범위)로 겁니다.

## 1단계 · 준비 — 라이브러리와 인증키

필요한 라이브러리는 6주차(confluent-kafka) · 7주차(minio · qdrant-client · sentence-transformers) · 5주차(pyspark)에서 쓴 것입니다. 없는 것만 설치합니다.

인증키는 서울 열린데이터광장에서 신청합니다(즉시 발급). 키가 없으면 공개 키 `sample` 로 진행합니다.
`sample` 은 두 가지가 다릅니다 — 실시간 도시데이터는 **장소가 「광화문·덕수궁」 하나로 고정**되고, 따릉이 실시간은 **5행까지만** 옵니다. 코드는 같고 결과의 양만 다릅니다.

In [ ]:
import sys, subprocess, importlib.util
def need(mod, pkg=None):
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg or mod], check=True)
need("confluent_kafka", "confluent-kafka")
need("minio")
need("qdrant_client", "qdrant-client")
need("sentence_transformers", "sentence-transformers")
need("pyspark", "pyspark==3.5.3")

import os, io, json, time, socket, shutil
from datetime import datetime
from zoneinfo import ZoneInfo
import requests
import pandas as pd

os.environ["TOKENIZERS_PARALLELISM"] = "false"          # 임베딩 모델과 하위 프로세스(docker) 를 함께 쓸 때 나는 경고 끄기
KST = ZoneInfo("Asia/Seoul")
SEOUL_KEY = os.environ.get("SEOUL_API_KEY") or "sample"     # 내 키는 환경 변수로 — 코드에 쓰지 않습니다
PLACES = ["강남역", "홍대 관광특구", "광화문·덕수궁"] if SEOUL_KEY != "sample" else ["광화문·덕수궁"]
print("인증키:", "내 키" if SEOUL_KEY != "sample" else "sample(장소 1곳 · 따릉이 5행)")
print("수집 장소:", PLACES)

print("docker:", shutil.which("docker") is not None, "· java:", shutil.which("java") is not None)
assert shutil.which("docker"), "Docker 가 없습니다 — Docker Desktop 을 설치하고 켜 주세요"

인증키: sample(장소 1곳 · 따릉이 5행)
수집 장소: ['광화문·덕수궁']
docker: True · java: True


> **확인 포인트** — `docker: True · java: True` 가 찍힙니다. java 는 6단계 Spark 가 씁니다(5주차).

> **Aebon's Tip** — 장소 이름은 서울 열린데이터광장의 「서울시 주요 장소 목록」 파일에 있는 이름 그대로 씁니다(예: `강남역`, `홍대 관광특구`). 장소 코드(`POI014` 같은)도 됩니다.

## 2단계 · 인프라 띄우기 — Kafka · MinIO · Qdrant (레이크 · Spark)

세 서버를 compose 파일 하나로 띄웁니다. Kafka 설정은 6주차 6-2 의 리스너 3개 구성 그대로입니다 — 내 노트북에서는 `localhost:9092`, 컨테이너끼리는 `kafka:29092` 로 붙습니다(6주차).
MinIO 는 과정에서 쓴 `pgsty/minio` 판(4주차), Qdrant 는 7주차의 벡터 DB 입니다. Airflow 는 9단계에서 검사할 때만 띄우도록 `profiles` 로 묶어 둡니다.

In [ ]:
%%writefile docker-compose.yml
services:
  kafka:
    image: apache/kafka:3.7.0
    ports:
      - "9092:9092"
    environment:
      KAFKA_NODE_ID: 1
      KAFKA_PROCESS_ROLES: broker,controller
      KAFKA_CONTROLLER_QUORUM_VOTERS: 1@kafka:29093
      KAFKA_LISTENERS: PLAINTEXT://:29092,CONTROLLER://:29093,EXTERNAL://:9092
      KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://kafka:29092,EXTERNAL://localhost:9092
      KAFKA_LISTENER_SECURITY_PROTOCOL_MAP: PLAINTEXT:PLAINTEXT,CONTROLLER:PLAINTEXT,EXTERNAL:PLAINTEXT
      KAFKA_CONTROLLER_LISTENER_NAMES: CONTROLLER
      KAFKA_INTER_BROKER_LISTENER_NAME: PLAINTEXT
      KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 1
      KAFKA_TRANSACTION_STATE_LOG_REPLICATION_FACTOR: 1
      KAFKA_TRANSACTION_STATE_LOG_MIN_ISR: 1
  minio:
    image: pgsty/minio:RELEASE.2026-08-04T00-00-00Z
    command: server /data --console-address ":9001"
    ports:
      - "9000:9000"
      - "9001:9001"
    environment:
      MINIO_ROOT_USER: minioadmin
      MINIO_ROOT_PASSWORD: minioadmin
    volumes:
      - minio-data:/data
  qdrant:
    image: qdrant/qdrant:v1.19.1
    ports:
      - "6333:6333"
    volumes:
      - qdrant-data:/qdrant/storage
  airflow:
    image: apache/airflow:2.9.3
    profiles: ["airflow"]
    volumes:
      - ./dags:/opt/airflow/dags
    environment:
      AIRFLOW__CORE__LOAD_EXAMPLES: "False"
      T1_KAFKA: kafka:29092
      T1_MINIO: minio:9000
      T1_QDRANT: http://qdrant:6333
volumes:
  minio-data:
  qdrant-data:

Writing docker-compose.yml


In [ ]:
!docker compose --progress quiet -p t1 up -d
!docker compose -p t1 ps --services --status running

kafka
minio
qdrant


`-p t1` 은 프로젝트 이름입니다. 컨테이너 · 볼륨 이름 앞에 `t1-` 이 붙어 다른 실습과 섞이지 않고, 마지막에 `down -v` 로 한 번에 지울 수 있습니다.
포트가 열렸는지와, Kafka 가 실제로 요청을 받는지까지 기다립니다.

In [ ]:
from confluent_kafka.admin import AdminClient, NewTopic
from confluent_kafka import KafkaException

def wait_port(port, host="localhost", sec=120):
    t0 = time.time()
    while time.time() - t0 < sec:
        try:
            socket.create_connection((host, port), timeout=2).close()
            return
        except OSError:
            time.sleep(1)
    raise TimeoutError(f"{host}:{port} 가 {sec}초 안에 열리지 않았습니다")

for p in (9092, 9000, 6333):
    wait_port(p)

BOOT = "localhost:9092"
KCONF = {"bootstrap.servers": BOOT, "log_level": 3}      # 3 = 오류만 찍기(기본 6 은 접속 재시도까지 모두 찍습니다)
admin = AdminClient(KCONF)
for _ in range(30):
    try:
        meta = admin.list_topics(timeout=5)
        break
    except KafkaException:
        time.sleep(2)
print("Kafka 브로커:", list(meta.brokers.values()))

Kafka 브로커: [BrokerMetadata(1, localhost:9092)]


> **확인 포인트** — `Kafka 브로커: [BrokerMetadata(1, localhost:9092)]` — 브로커 1번이 내 노트북용 주소(EXTERNAL)로 안내됩니다.

In [ ]:
assert len(meta.brokers) == 1 and "localhost:9092" in str(meta.brokers[1])

일부러 틀린 포트로 붙어 보세요. 브로커가 없으면 기다리다가 `KafkaException` 이 납니다.

In [ ]:
try:
    AdminClient({"bootstrap.servers": "localhost:9999", "log_level": 0}).list_topics(timeout=3)   # 0 = 재시도 로그도 끄기
except KafkaException as e:
    print("접속 실패:", str(e)[:70])

접속 실패: KafkaError{code=_TRANSPORT,val=-195,str="Failed to get metadata: Local


## 3단계 · 수집 — 실시간 도시데이터 한 건 뜯어보기 (수집 · 스트리밍)

서울 OpenAPI 주소는 `http://openapi.seoul.go.kr:8088/<키>/json/<서비스>/<시작>/<끝>/<장소>` 꼴입니다(11장 크롤러에서 쓴 requests 그대로).
응답의 `RESULT.CODE` 가 `INFO-000` 이 아니면 실패입니다 — HTTP 는 200 인데 내용이 오류인 경우가 많으니 **코드를 꼭 확인**합니다.

In [ ]:
API = "http://openapi.seoul.go.kr:8088"
UA = {"User-Agent": "Mozilla/5.0 (Macintosh)"}

def call(service, start=1, end=5, *rest):
    url = "/".join([API, SEOUL_KEY, "json", service, str(start), str(end), *rest])
    r = requests.get(url, headers=UA, timeout=30)
    r.raise_for_status()
    d = r.json()
    res = d.get("RESULT") or next((v.get("RESULT") for v in d.values() if isinstance(v, dict) and "RESULT" in v), {})
    code = res.get("CODE") or res.get("RESULT.CODE")
    if code != "INFO-000":
        raise RuntimeError(f"{service}: {code} {res.get('MESSAGE') or res.get('RESULT.MESSAGE')}")
    return d

city = call("citydata", 1, 5, PLACES[0])["CITYDATA"]
print(city["AREA_NM"], city["AREA_CD"])
print([k for k in city if isinstance(city[k], list)])

광화문·덕수궁 POI009
['LIVE_PPLTN_STTS', 'PRK_STTS', 'SUB_STTS', 'BUS_STN_STTS', 'ACDNT_CNTRL_STTS', 'SBIKE_STTS', 'WEATHER_STTS', 'CHARGER_STTS', 'EVENT_STTS', 'LIVE_DST_MESSAGE', 'LIVE_YNA_NEWS']


한 건 안에 목록이 여럿 들어 있습니다. 그중 이 프로젝트가 쓰는 것은 다섯 가지입니다.

| 항목 | 내용 | 쓰는 곳 |
|---|---|---|
| `LIVE_PPLTN_STTS` | 인구 혼잡도 · 인구 범위 · 12시간 예측 | Spark 집계 · RAG |
| `ROAD_TRAFFIC_STTS` | 주변 도로 소통(평균 속도) | Spark · RAG |
| `SBIKE_STTS` | 주변 따릉이 대여소의 거치 대수 | Spark |
| `EVENT_STTS` · `ACDNT_CNTRL_STTS` | 문화행사 · 사고 · 공사 통제 | RAG 문서 |
| `WEATHER_STTS` | 기온 · 강수 · 대기 안내 글 | RAG 문서 |

In [ ]:
pp = city["LIVE_PPLTN_STTS"][0]
print(pp["PPLTN_TIME"], pp["AREA_CONGEST_LVL"], pp["AREA_PPLTN_MIN"], "~", pp["AREA_PPLTN_MAX"])
print("예측", len(pp["FCST_PPLTN"]), "개 시각 · 행사", len(city["EVENT_STTS"] or []), "건 · 통제", len(city["ACDNT_CNTRL_STTS"] or []), "건")

2026-10-28 22:00 여유 12000 ~ 14000
예측 12 개 시각 · 행사 13 건 · 통제 1 건


> **확인 포인트** — 장소 이름과 `AREA_CD`(예: `광화문·덕수궁 POI009`), 그리고 혼잡도가 `여유 · 보통 · 약간 붐빔 · 붐빔` 넷 중 하나로 찍힙니다.

In [ ]:
assert city["AREA_CD"].startswith("POI")
assert pp["AREA_CONGEST_LVL"] in ("여유", "보통", "약간 붐빔", "붐빔")
assert int(pp["AREA_PPLTN_MIN"]) <= int(pp["AREA_PPLTN_MAX"])

따릉이 실시간 대여정보(`bikeList`)도 같은 주소 규칙입니다. `sample` 키는 5행까지만 줍니다. 내 키가 있으면 1~1000 · 1001~2000 처럼 1,000행씩 나눠 받아 서울 전체 2천여 곳을 모읍니다.

In [ ]:
def bike_rows():
    if SEOUL_KEY == "sample":
        return call("bikeList", 1, 5)["rentBikeStatus"]["row"]
    rows, start = [], 1
    while True:                                   # 내 키 — 한 번에 최대 1,000행
        part = call("bikeList", start, start + 999)["rentBikeStatus"]["row"]
        rows += part
        if len(part) < 1000:
            return rows
        start += 1000

bikes = pd.DataFrame(bike_rows())
print(bikes.shape)
bikes[["stationId", "stationName", "parkingBikeTotCnt", "rackTotCnt"]].head()

(5, 7)


,stationId,stationName,parkingBikeTotCnt,rackTotCnt
0,ST-4,102. 망원역 1번출구 앞,5,15
1,ST-5,103. 망원역 2번출구 앞,6,14
2,ST-6,104. 합정역 1번출구 앞,2,13
3,ST-7,105. 합정역 5번출구 앞,2,5
4,ST-8,106. 합정역 7번출구 앞,1,12


> **확인 포인트** — `sample` 이면 `(5, 7)` — 대여소 5곳, 열 7개(`stationId` · `parkingBikeTotCnt` 등)입니다.

In [ ]:
assert len(bikes) >= 5 and {"stationId", "parkingBikeTotCnt", "rackTotCnt"} <= set(bikes.columns)

일부러 서비스 이름을 틀려 보세요. HTTP 는 성공인데 응답 코드가 오류입니다.

In [ ]:
try:
    call("citydat", 1, 5, PLACES[0])
except RuntimeError as e:
    print("수집 실패:", str(e)[:60])

수집 실패: citydat: ERROR-500 서버 오류입니다.
지속적으로 발생시 열린 데이터 광장으로 문의(Q&A) 바


## 4단계 · Kafka Producer — 응답을 토픽으로 흘리기 (수집 · 스트리밍)

응답 전체(약 170KB)를 그대로 흘리지 않고, 쓸 항목만 골라 작은 메시지로 만듭니다. 문자로 온 숫자(`"14000"`)도 여기서 정수로 바꿉니다.
메시지 키는 장소 코드입니다 — 같은 키는 같은 파티션으로 가서 한 장소의 순서가 지켜집니다(6주차).

> **자주 겪는 오류** — 실시간 도시데이터 안의 `LIVE_YNA_NEWS`(연합뉴스 기사)는 언론사 저작물이라 공공누리 대상이 아닙니다. 이 프로젝트에서는 담지 않습니다.

In [ ]:
def to_int(x):
    try:
        return int(float(x))
    except (TypeError, ValueError):
        return None

def slim(c):
    """citydata 한 건 → Kafka 로 보낼 작은 메시지(dict)."""
    p = c["LIVE_PPLTN_STTS"][0]
    road = (c.get("ROAD_TRAFFIC_STTS") or {}).get("AVG_ROAD_DATA") or {}
    w = (c.get("WEATHER_STTS") or [{}])[0]
    return {
        "area": c["AREA_NM"], "area_cd": c["AREA_CD"],
        "collected_at": datetime.now(KST).strftime("%Y-%m-%d %H:%M:%S"),
        "ppltn_time": p["PPLTN_TIME"], "congest_lvl": p["AREA_CONGEST_LVL"], "congest_msg": p["AREA_CONGEST_MSG"],
        "ppltn_min": to_int(p["AREA_PPLTN_MIN"]), "ppltn_max": to_int(p["AREA_PPLTN_MAX"]),
        "fcst": [{"time": f["FCST_TIME"], "lvl": f["FCST_CONGEST_LVL"], "max": to_int(f["FCST_PPLTN_MAX"])}
                 for f in (p.get("FCST_PPLTN") or [])],
        "road_idx": road.get("ROAD_TRAFFIC_IDX"), "road_spd": to_int(road.get("ROAD_TRAFFIC_SPD")), "road_msg": road.get("ROAD_MSG"),
        "bike_parking": sum(to_int(b["SBIKE_PARKING_CNT"]) or 0 for b in (c.get("SBIKE_STTS") or [])),
        "bike_racks": sum(to_int(b["SBIKE_RACK_CNT"]) or 0 for b in (c.get("SBIKE_STTS") or [])),
        "events": [{"name": e["EVENT_NM"], "period": e["EVENT_PERIOD"], "place": (e.get("EVENT_PLACE") or "").strip()}
                   for e in (c.get("EVENT_STTS") or [])],
        "accidents": [{"type": a["ACDNT_TYPE"], "dtype": a["ACDNT_DTYPE"], "info": a["ACDNT_INFO"], "until": a["EXP_CLR_DT"]}
                      for a in (c.get("ACDNT_CNTRL_STTS") or [])],
        "weather": {"time": w.get("WEATHER_TIME"), "temp": w.get("TEMP"), "pcp_msg": w.get("PCP_MSG"), "air_msg": w.get("AIR_MSG")},
    }

msg = slim(city)
print(len(json.dumps(msg, ensure_ascii=False).encode()), "bytes")
{k: msg[k] for k in ("area", "ppltn_time", "congest_lvl", "ppltn_min", "ppltn_max", "road_spd", "bike_parking")}

3778 bytes


{'area': '광화문·덕수궁',
 'ppltn_time': '2026-10-28 22:00',
 'congest_lvl': '여유',
 'ppltn_min': 12000,
 'ppltn_max': 14000,
 'road_spd': 20,
 'bike_parking': 89}

토픽 두 개를 만들고, 수집을 세 번 되풀이해 보냅니다. 실제 운영(9단계)에서는 Airflow 가 매시간, 또는 Producer 가 1분마다 이 일을 합니다.

In [ ]:
from confluent_kafka import Producer

admin.create_topics([NewTopic("city.live", num_partitions=1, replication_factor=1),
                     NewTopic("bike.live", num_partitions=1, replication_factor=1)])
time.sleep(2)

producer = Producer(KCONF)
sent = {"city.live": 0, "bike.live": 0}

def on_done(err, m):
    if err is None:
        sent[m.topic()] += 1

ROUNDS = 3
for r in range(ROUNDS):
    for place in PLACES:
        m = slim(call("citydata", 1, 5, place)["CITYDATA"])
        producer.produce("city.live", key=m["area_cd"], value=json.dumps(m, ensure_ascii=False), callback=on_done)
    if r == 0:                                     # 따릉이는 한 번만(1,000행 단위라 자주 받을 필요가 없습니다)
        for b in bike_rows():
            producer.produce("bike.live", key=b["stationId"], value=json.dumps(b, ensure_ascii=False), callback=on_done)
    producer.flush()
    time.sleep(2)
print(sent)

{'city.live': 3, 'bike.live': 5}


> **확인 포인트** — `sample` 이면 `{'city.live': 3, 'bike.live': 5}` — 장소 1곳 × 3번, 따릉이 5행입니다. 내 키면 장소 수 × 3 과 따릉이 2천여 행입니다.

In [ ]:
N_CITY, N_BIKE = len(PLACES) * ROUNDS, len(bikes)
assert sent == {"city.live": N_CITY, "bike.live": N_BIKE}

일부러 dict 를 그대로 보내 보세요. Kafka 메시지는 바이트(또는 글자)여야 합니다.

In [ ]:
try:
    producer.produce("city.live", value=msg)
except TypeError as e:
    print("보내기 실패:", str(e)[:60])

보내기 실패: a bytes-like object is required, not 'dict'


## 5단계 · Consumer → MinIO — 레이크의 원본 층에 쌓기 (수집 · 스트리밍)

Consumer 는 두 토픽을 구독해 메시지 하나를 JSON 파일 하나로 MinIO 에 씁니다(17장 MinIO).
경로를 `dt=날짜/hour=시` 로 나누는 것이 핵심입니다 — 이 이름 규칙(하이브 파티션)을 Spark 가 읽어 `dt` · `hour` 열로 바꿔 줍니다(5주차).

In [ ]:
from minio import Minio
from minio.error import S3Error
from confluent_kafka import Consumer

mc = Minio("localhost:9000", access_key="minioadmin", secret_key="minioadmin", secure=False)
if not mc.bucket_exists("lake"):
    mc.make_bucket("lake")

consumer = Consumer({**KCONF, "group.id": "t1-lake",
                     "auto.offset.reset": "earliest", "enable.auto.commit": False})
consumer.subscribe(["city.live", "bike.live"])

written, idle = [], 0
while idle < 10:                                  # 10초 동안 새 메시지가 없으면 끝
    m = consumer.poll(1.0)
    if m is None:
        idle += 1
        continue
    if m.error():
        raise KafkaException(m.error())
    idle = 0
    body = m.value()
    at = json.loads(body).get("collected_at") or datetime.now(KST).strftime("%Y-%m-%d %H:%M:%S")
    kind = m.topic().split(".")[0]                         # city · bike
    key = f"raw/{kind}/dt={at[:10]}/hour={at[11:13]}/{kind}-p{m.partition()}-o{m.offset()}.json"
    mc.put_object("lake", key, io.BytesIO(body), len(body), content_type="application/json")
    written.append(key)
consumer.commit(asynchronous=False)
consumer.close()

print(len(written), "개 파일")
print(written[0])

8 개 파일
raw/city/dt=2026-10-28/hour=22/city-p0-o0.json


`enable.auto.commit` 을 끄고 파일을 다 쓴 뒤에 커밋했습니다. 쓰다가 죽으면 커밋하지 않은 메시지를 다음에 다시 읽습니다 — 잃어버리지는 않고, 중복은 6단계에서 걸러냅니다(최소 한 번 전달, 6주차).

> **확인 포인트** — 보낸 수와 같은 수(`sample` 이면 8개)의 파일이 생기고, 경로가 `raw/city/dt=2026-…/hour=…/city-p0-o0.json` 꼴입니다.

In [ ]:
objs = [o.object_name for o in mc.list_objects("lake", prefix="raw/", recursive=True)]
assert len(written) == N_CITY + N_BIKE and len(objs) == N_CITY + N_BIKE
assert sum("raw/city/dt=" in o for o in objs) == N_CITY

> **Aebon's Tip** — 6주차 6-5 에서는 같은 일을 **Kafka Connect 의 S3 Sink 커넥터**로 했습니다(코드 없이 JSON 설정만). 커넥터는 플러그인을 따로 받아야 해서 이 노트북은 Consumer 코드로 썼습니다.
> 운영형으로 바꾼다면 `flush.size` · `path.format`(`'dt'=YYYY-MM-dd/'hour'=HH`)을 설정한 S3 Sink 로 옮기세요. 레이크의 경로 규칙만 같으면 6단계부터는 그대로 돕니다.

일부러 없는 버킷에 써 보세요.

In [ ]:
try:
    mc.put_object("no-such-bucket", "x.json", io.BytesIO(b"{}"), 2)
except S3Error as e:
    print("쓰기 실패:", e.code)

쓰기 실패: NoSuchBucket


## 6단계 · Spark 혼잡도 — 원본 층에서 마트 표 만들기 (레이크 · Spark)

Spark 가 MinIO 를 `s3a://` 주소로 읽으려면 hadoop-aws 라이브러리와 접속 설정 네 줄이 필요합니다(5주차 — 「길」 라이브러리). 처음 실행하면 라이브러리(약 300MB)를 받느라 1~2분 걸립니다.

In [ ]:
from pyspark.sql import SparkSession, functions as F, Window

spark = (SparkSession.builder.appName("t1-congestion").master("local[2]")
         .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4")
         .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000")
         .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
         .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
         .config("spark.hadoop.fs.s3a.path.style.access", "true")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

raw = spark.read.json("s3a://lake/raw/city/")
print(raw.count(), "행")
raw.select("dt", "hour", "area", "ppltn_time", "congest_lvl", "ppltn_min", "ppltn_max").show(truncate=False)

:: loading settings :: url = jar:file:…


3 행


+----------+----+-------------+----------------+-----------+---------+---------+
|dt        |hour|area         |ppltn_time      |congest_lvl|ppltn_min|ppltn_max|
+----------+----+-------------+----------------+-----------+---------+---------+
|2026-10-28|22  |광화문·덕수궁|2026-10-28 22:00|여유       |12000    |14000    |
|2026-10-28|22  |광화문·덕수궁|2026-10-28 22:00|여유       |12000    |14000    |
|2026-10-28|22  |광화문·덕수궁|2026-10-28 22:00|여유       |12000    |14000    |
+----------+----+-------------+----------------+-----------+---------+---------+



`dt` · `hour` 는 파일 안에 없던 열입니다. 5단계의 경로 이름에서 Spark 가 만들어 낸 것입니다.

같은 시각(`ppltn_time`) 자료를 여러 번 받았습니다 — 서울시는 5분마다 갱신하는데 우리는 몇 초 간격으로 세 번 받았기 때문입니다. 장소 · 시각이 같으면 한 번만 셉니다.

In [ ]:
snap = raw.dropDuplicates(["area_cd", "ppltn_time"])
print("원본", raw.count(), "→ 중복 제거", snap.count())

LEVEL = F.when(F.col("congest_lvl") == "여유", 1).when(F.col("congest_lvl") == "보통", 2) \
         .when(F.col("congest_lvl") == "약간 붐빔", 3).when(F.col("congest_lvl") == "붐빔", 4)

w = Window.partitionBy("area_cd").orderBy(F.col("ppltn_time").desc())
latest = (snap.withColumn("r", F.row_number().over(w)).where("r = 1")
          .withColumn("level", LEVEL)
          .withColumn("ppltn_mid", (F.col("ppltn_min") + F.col("ppltn_max")) / 2)
          .withColumn("bike_rate", F.round(F.col("bike_parking") / F.col("bike_racks"), 2)))

peak = (snap.select("area_cd", F.explode("fcst").alias("f"))
        .groupBy("area_cd").agg(F.max_by("f.time", "f.max").alias("peak_time"), F.max("f.max").alias("peak_max")))

mart = latest.join(peak, "area_cd", "left").select(
    "area", "area_cd", "ppltn_time", "congest_lvl", "level", "ppltn_mid", "road_idx", "road_spd",
    "bike_parking", "bike_racks", "bike_rate", "peak_time", "peak_max")
mart.show(truncate=False)

원본 3 → 중복 제거 1


+-------------+-------+----------------+-----------+-----+---------+--------+--------+------------+----------+---------+----------------+--------+
|area         |area_cd|ppltn_time      |congest_lvl|level|ppltn_mid|road_idx|road_spd|bike_parking|bike_racks|bike_rate|peak_time       |peak_max|
+-------------+-------+----------------+-----------+-----+---------+--------+--------+------------+----------+---------+----------------+--------+
|광화문·덕수궁|POI009 |2026-10-28 22:00|여유       |1    |13000.0  |서행    |20      |89          |121       |0.74     |2026-10-29 10:00|36000   |
+-------------+-------+----------------+-----------+-----+---------+--------+--------+------------+----------+---------+----------------+--------+



「장소마다 가장 최근 한 건」은 윈도 함수 `row_number()` 로 풉니다(2주차 8장과 같은 생각을 Spark 로). 예측 12시간은 `explode` 로 행을 펼친 뒤 가장 붐빌 때를 찾습니다.

> **확인 포인트** — 중복 제거 뒤 행 수가 원본보다 많지 않고(`sample` 이면 원본 3 → 1~2), 마트는 장소마다 한 행이며 `level` 은 1~4 입니다.

In [ ]:
rows = mart.collect()
assert raw.count() == N_CITY and 1 <= snap.count() <= N_CITY
assert len(rows) == len(PLACES) and all(1 <= r["level"] <= 4 for r in rows)
assert all(r["ppltn_mid"] > 0 for r in rows)

마트 표는 Parquet 으로 레이크의 다른 층(`mart/`)에 씁니다. 원본(`raw/`)은 건드리지 않습니다 — 다시 계산하고 싶을 때 원본이 남아 있어야 합니다.

In [ ]:
mart.write.mode("overwrite").parquet("s3a://lake/mart/congestion/")
back = spark.read.parquet("s3a://lake/mart/congestion/")
print(back.count(), back.columns[:5])
assert back.count() == len(PLACES)

1 ['area', 'area_cd', 'ppltn_time', 'congest_lvl', 'level']


일부러 없는 열을 골라 보세요. Spark 는 실행 전에 열 이름부터 검사합니다.

In [ ]:
from pyspark.errors import AnalysisException
try:
    raw.select("congest_level").show()
except AnalysisException as e:
    print("분석 실패:", str(e).splitlines()[0][:70])

분석 실패: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter wit


## 7단계 · RAG 문서 만들기 — 글을 Qdrant 에 넣기 (RAG · 서비스)

같은 응답 속의 글을 근거 문서로 바꿉니다. 문서 하나 = 사실 하나가 되게 잘게 나눕니다(7주차 청킹). 행사 하나, 통제 하나, 날씨 안내 하나가 각각 문서입니다.
메타데이터(`area` · `kind`)를 함께 넣어, 검색할 때 「광화문 문서만」 고를 수 있게 합니다.

In [ ]:
def docs_from(m):
    a, out = m["area"], []
    out.append(("congestion", f"{a} 인구 혼잡도는 {m['congest_lvl']}입니다(약 {m['ppltn_min']:,}~{m['ppltn_max']:,}명, {m['ppltn_time']} 기준). {m['congest_msg']}"))
    if m["fcst"]:
        top = max(m["fcst"], key=lambda f: f["max"] or 0)
        out.append(("forecast", f"{a} 앞으로 12시간 중 가장 붐빌 때는 {top['time']}, 예상 혼잡도 {top['lvl']}(최대 약 {top['max']:,}명)입니다."))
    if m["road_idx"]:
        out.append(("road", f"{a} 주변 도로는 {m['road_idx']}, 평균 시속 {m['road_spd']}km 입니다. {m['road_msg'] or ''}"))
    for e in m["events"]:
        out.append(("event", f"{a} 문화행사: {e['name']} — 기간 {e['period']}, 장소 {e['place']}"))
    for x in m["accidents"]:
        out.append(("accident", f"{a} 사고 · 통제({x['type']}/{x['dtype']}): {x['info']} — 해제 예정 {x['until']}"))
    wt = m["weather"]
    out.append(("weather", f"{a} 날씨({wt['time']}): 기온 {wt['temp']}도. {wt['pcp_msg'] or ''} {wt['air_msg'] or ''}"))
    return [{"area": a, "kind": k, "text": t, "at": m["ppltn_time"]} for k, t in out]

latest_msgs = {}
for o in mc.list_objects("lake", prefix="raw/city/", recursive=True):
    m = json.loads(mc.get_object("lake", o.object_name).read())
    if m["area_cd"] not in latest_msgs or m["ppltn_time"] >= latest_msgs[m["area_cd"]]["ppltn_time"]:
        latest_msgs[m["area_cd"]] = m

docs = [d for m in latest_msgs.values() for d in docs_from(m)]
print(len(docs), "개 문서")
pd.Series([d["kind"] for d in docs]).value_counts()

18 개 문서


event         13
congestion     1
forecast       1
road           1
accident       1
weather        1
Name: count, dtype: int64

> **확인 포인트** — 장소마다 `congestion` · `weather` 문서가 하나씩은 꼭 있고, 행사(`event`)는 그날 열리는 만큼 생깁니다(광화문·덕수궁은 보통 열 건 남짓).

In [ ]:
kinds = [d["kind"] for d in docs]
assert kinds.count("congestion") == len(PLACES) and kinds.count("weather") == len(PLACES)

임베딩은 키 없이 도는 한국어 모델 `jhgan/ko-sroberta-multitask`(768차원)를 씁니다. 처음에는 모델(약 450MB)을 받느라 1분쯤 걸립니다.

In [ ]:
import re, uuid
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue

emb = SentenceTransformer("jhgan/ko-sroberta-multitask")
DIM = emb.get_sentence_embedding_dimension()
qd = QdrantClient(url="http://localhost:6333")

if qd.collection_exists("t1_city"):
    qd.delete_collection("t1_city")
qd.create_collection("t1_city", vectors_config=VectorParams(size=DIM, distance=Distance.COSINE))

vecs = emb.encode([d["text"] for d in docs], normalize_embeddings=True)
points = [PointStruct(id=str(uuid.uuid5(uuid.NAMESPACE_URL, d["area"] + d["kind"] + d["text"])), vector=v.tolist(), payload=d)
          for d, v in zip(docs, vecs)]
qd.upsert("t1_city", points=points)
print(DIM, "차원 ·", qd.count("t1_city").count, "개 저장")

768 차원 · 18 개 저장


문서 id 를 글 내용에서 만들었습니다(`uuid5`). 매시간 다시 넣어도 같은 문서는 덮어써져 늘어나지 않습니다.

In [ ]:
assert DIM == 768 and qd.count("t1_city").count == len(docs)

일부러 차원이 다른 벡터를 넣어 보세요. 컬렉션을 만들 때 정한 크기와 다르면 거절됩니다.

In [ ]:
def qerr(e):
    """Qdrant 오류 응답에서 사람이 읽을 부분만 꺼냅니다."""
    m = re.search(r'"error":"([^"]+)', str(e))
    return m.group(1) if m else str(e)[:80]

try:
    qd.upsert("t1_city", points=[PointStruct(id=1, vector=[0.1] * 384, payload={})])
except Exception as e:
    print("넣기 실패:", qerr(e))

넣기 실패: Wrong input: Vector dimension error: expected dim: 768, got 384


## 8단계 · 「지금 광화문 왜 붐벼?」 — 숫자와 글을 함께 붙이기 (RAG · 서비스)

질문 하나에 두 가지를 찾습니다. **숫자**는 6단계의 마트 표에서(지금 혼잡도 · 가장 붐빌 시각), **이유가 될 만한 글**은 Qdrant 에서(행사 · 통제 · 도로).
질문에서 장소를 먼저 알아내 검색을 그 장소로 좁힙니다 — 다른 장소의 행사가 근거로 섞이지 않게 하는 것입니다.

In [ ]:
import re
mart_pd = back.toPandas()

def find_area(q):
    for a in mart_pd["area"]:
        if any(t[:2] in q for t in re.split(r"[·\s]", a) if t):
            return a
    return mart_pd["area"].iloc[0]

def ask(q, k=4):
    area = find_area(q)
    row = mart_pd[mart_pd["area"] == area].iloc[0].to_dict()
    hits = qd.query_points("t1_city", query=emb.encode(q, normalize_embeddings=True).tolist(), limit=k,
                           query_filter=Filter(must=[FieldCondition(key="area", match=MatchValue(value=area))])).points
    return area, row, hits

Q = "지금 광화문 왜 붐벼?"
area, row, hits = ask(Q)
print(f"[숫자] {area} · {row['congest_lvl']}(level {row['level']}) · 약 {int(row['ppltn_mid']):,}명 · 가장 붐빌 때 {row['peak_time']}")
for h in hits:
    print(f"[글 {h.score:.2f}] ({h.payload['kind']}) {h.payload['text'][:70]}")

[숫자] 광화문·덕수궁 · 여유(level 1) · 약 13,000명 · 가장 붐빌 때 2026-10-29 10:00
[글 0.64] (forecast) 광화문·덕수궁 앞으로 12시간 중 가장 붐빌 때는 2026-10-29 10:00, 예상 혼잡도 보통(최대 약 36,000명)입
[글 0.59] (event) 광화문·덕수궁 문화행사: [서울시청] 2026 운동하는 서울광장 — 기간 2026-09-01~2026-10-30, 장소 서울광
[글 0.54] (event) 광화문·덕수궁 문화행사: 2026 문화가 흐르는 서울광장 — 기간 2026-05-06~2026-12-31, 장소 서울광장
[글 0.52] (event) 광화문·덕수궁 문화행사: [서울야외도서관] 2026년 서울야외도서관 (책읽는 서울광장， 광화문 책마당， 책읽는 맑은냇가) — 


> **확인 포인트** — 장소가 `광화문·덕수궁` 으로 잡히고, 글 근거가 1건 이상 나오며 모두 같은 장소 문서입니다. 무엇이 1위인지는 그때의 행사 · 통제에 따라 달라집니다.

In [ ]:
assert area == "광화문·덕수궁" or SEOUL_KEY != "sample"
assert 1 <= len(hits) <= 4 and all(h.payload["area"] == area for h in hits)

답 문장은 OpenAI `gpt-4o-mini` 가 만듭니다(7주차). **키가 있을 때만** 부르고, 없으면 위의 근거를 보여 주는 것으로 끝납니다. 근거 밖의 이야기를 지어내지 않도록 지시문에 못 박습니다.

In [ ]:
OPENAI_KEY = os.environ.get("OPENAI_API_KEY")
context = f"[현재 수치] {area} 혼잡도 {row['congest_lvl']}, 약 {int(row['ppltn_mid']):,}명, 도로 {row['road_idx']} {row['road_spd']}km/h, 가장 붐빌 예상 시각 {row['peak_time']}\n" + \
          "\n".join(f"[문서 {i}] {h.payload['text']}" for i, h in enumerate(hits, 1))

if OPENAI_KEY:
    need("openai")
    from openai import OpenAI
    res = OpenAI(api_key=OPENAI_KEY).chat.completions.create(model="gpt-4o-mini", temperature=0, messages=[
        {"role": "system", "content": "서울 도시 상황 안내원입니다. 아래 근거만으로 답하고, 근거에 없으면 모른다고 답하세요. 문장 끝에 [문서 n] 을 붙이세요."},
        {"role": "user", "content": context + "\n\n질문: " + Q}])
    print(res.choices[0].message.content)
else:
    print("OPENAI_API_KEY 가 없어 근거만 보여 줍니다.\n")
    print(context)

OPENAI_API_KEY 가 없어 근거만 보여 줍니다.

[현재 수치] 광화문·덕수궁 혼잡도 여유, 약 13,000명, 도로 서행 20km/h, 가장 붐빌 예상 시각 2026-10-29 10:00
[문서 1] 광화문·덕수궁 앞으로 12시간 중 가장 붐빌 때는 2026-10-29 10:00, 예상 혼잡도 보통(최대 약 36,000명)입니다.
[문서 2] 광화문·덕수궁 문화행사: [서울시청] 2026 운동하는 서울광장 — 기간 2026-09-01~2026-10-30, 장소 서울광장
[문서 3] 광화문·덕수궁 문화행사: 2026 문화가 흐르는 서울광장 — 기간 2026-05-06~2026-12-31, 장소 서울광장
[문서 4] 광화문·덕수궁 문화행사: [서울야외도서관] 2026년 서울야외도서관 (책읽는 서울광장， 광화문 책마당， 책읽는 맑은냇가) — 기간 2026-09-04~2026-11-01, 장소 서울광장， 광화문광장(육조마당)， 청계천(모전교~광통교)


> **Aebon's Tip** — 키 없이 답 문장까지 만들고 싶으면 7주차의 Ollama(로컬 LLM)로 바꾸세요. `context` 를 만드는 앞부분은 그대로 쓰고, 마지막 호출만 바뀝니다.

일부러 장소를 거르지 않고 검색해 보세요. 내 키로 여러 장소를 모았다면 다른 장소의 행사가 섞여 나옵니다. `sample` 은 장소가 하나라 결과가 같습니다 — 그래서 필터가 「있어도 없어도 같아 보이는」 코드가 되기 쉽습니다.
이어서 길이가 틀린 질문 벡터(10차원)로 검색해 실패도 보세요.

In [ ]:
try:
    loose = qd.query_points("t1_city", query=emb.encode(Q, normalize_embeddings=True).tolist(), limit=4).points
    print("필터 없이:", sorted({h.payload["area"] for h in loose}))
    qd.query_points("t1_city", query=[0.0] * 10, limit=1)
except Exception as e:
    print("검색 실패:", qerr(e))

필터 없이: ['광화문·덕수궁']
검색 실패: Wrong input: Vector dimension error: expected dim: 768, got 10


## 9단계 · Airflow DAG — 매시간 네 일을 차례로 (오케스트레이션)

지금까지 손으로 한 일을 함수 네 개로 묶어 `dags/t1_jobs.py` 에 두고, DAG 파일이 그 함수를 매시간 차례로 부르게 합니다(8주차).
접속 주소는 환경 변수로 받습니다 — 노트북에서는 `localhost:9092`, Airflow 컨테이너 안에서는 `kafka:29092` 입니다(2단계 compose 의 `T1_KAFKA`).

In [ ]:
os.makedirs("dags", exist_ok=True)

In [ ]:
%%writefile dags/t1_jobs.py
"""T1 서울 실시간 도시 상황판 — Airflow 가 매시간 부르는 네 가지 일(노트북 3~7단계를 함수로 묶은 것)."""
import io, json, os, time, uuid
from datetime import datetime
from zoneinfo import ZoneInfo

KAFKA = os.environ.get("T1_KAFKA", "localhost:9092")
MINIO = os.environ.get("T1_MINIO", "localhost:9000")
QDRANT = os.environ.get("T1_QDRANT", "http://localhost:6333")
KEY = os.environ.get("SEOUL_API_KEY") or "sample"
PLACES = ["강남역", "홍대 관광특구", "광화문·덕수궁"] if KEY != "sample" else ["광화문·덕수궁"]
KST = ZoneInfo("Asia/Seoul")


def collect_produce(slim):
    """① 수집 · 스트리밍 — citydata 를 받아 city.live 토픽으로."""
    import requests
    from confluent_kafka import Producer
    p = Producer({"bootstrap.servers": KAFKA, "log_level": 3})
    for place in PLACES:
        url = f"http://openapi.seoul.go.kr:8088/{KEY}/json/citydata/1/5/{place}"
        c = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30).json()["CITYDATA"]
        m = slim(c)
        p.produce("city.live", key=m["area_cd"], value=json.dumps(m, ensure_ascii=False))
    return p.flush(30)


def consume_to_lake():
    """② 수집 · 스트리밍 — 토픽을 읽어 MinIO lake/raw 에 파일로."""
    from confluent_kafka import Consumer
    from minio import Minio
    mc = Minio(MINIO, access_key="minioadmin", secret_key="minioadmin", secure=False)
    c = Consumer({"bootstrap.servers": KAFKA, "log_level": 3, "group.id": "t1-lake", "auto.offset.reset": "earliest", "enable.auto.commit": False})
    c.subscribe(["city.live"])
    n, idle = 0, 0
    while idle < 10:
        m = c.poll(1.0)
        if m is None:
            idle += 1
            continue
        idle, body = 0, m.value()
        at = json.loads(body)["collected_at"]
        key = f"raw/city/dt={at[:10]}/hour={at[11:13]}/city-p{m.partition()}-o{m.offset()}.json"
        mc.put_object("lake", key, io.BytesIO(body), len(body), content_type="application/json")
        n += 1
    c.commit(asynchronous=False)
    c.close()
    return n


def spark_congestion(spark):
    """③ 레이크 · Spark — raw → mart/congestion (6단계와 같은 계산)."""
    from pyspark.sql import functions as F, Window
    snap = spark.read.json("s3a://lake/raw/city/").dropDuplicates(["area_cd", "ppltn_time"])
    w = Window.partitionBy("area_cd").orderBy(F.col("ppltn_time").desc())
    latest = snap.withColumn("r", F.row_number().over(w)).where("r = 1").drop("r")
    latest.select("area", "area_cd", "ppltn_time", "congest_lvl", "ppltn_min", "ppltn_max", "road_idx", "road_spd") \
          .write.mode("overwrite").parquet("s3a://lake/mart/congestion_hourly/")
    return latest.count()


def refresh_rag(docs_from, emb=None):
    """④ RAG · 서비스 — 장소마다 최신 메시지로 문서를 다시 만들어 Qdrant 를 갈아 끼우기."""
    from minio import Minio
    from qdrant_client import QdrantClient
    from qdrant_client.models import PointStruct, Filter, FieldCondition, MatchValue, FilterSelector
    if emb is None:
        from sentence_transformers import SentenceTransformer
        emb = SentenceTransformer("jhgan/ko-sroberta-multitask")
    mc = Minio(MINIO, access_key="minioadmin", secret_key="minioadmin", secure=False)
    latest = {}
    for o in mc.list_objects("lake", prefix="raw/city/", recursive=True):
        m = json.loads(mc.get_object("lake", o.object_name).read())
        if m["area_cd"] not in latest or m["ppltn_time"] >= latest[m["area_cd"]]["ppltn_time"]:
            latest[m["area_cd"]] = m
    qd = QdrantClient(url=QDRANT)
    n = 0
    for m in latest.values():
        docs = docs_from(m)
        qd.delete("t1_city", points_selector=FilterSelector(filter=Filter(must=[FieldCondition(key="area", match=MatchValue(value=m["area"]))])))
        vecs = emb.encode([d["text"] for d in docs], normalize_embeddings=True)
        qd.upsert("t1_city", points=[PointStruct(id=str(uuid.uuid5(uuid.NAMESPACE_URL, d["area"] + d["kind"] + d["text"])),
                                                 vector=v.tolist(), payload=d) for d, v in zip(docs, vecs)])
        n += len(docs)
    return n, qd.count("t1_city").count

Writing dags/t1_jobs.py


In [ ]:
%%writefile dags/t1_city_hourly.py
from datetime import datetime, timedelta
from airflow import DAG
from airflow.operators.python import PythonOperator

default_args = {"owner": "t1", "retries": 2, "retry_delay": timedelta(minutes=5)}

with DAG(
    dag_id="t1_city_hourly",
    start_date=datetime(2026, 11, 1),
    schedule="0 * * * *",
    catchup=False,
    default_args=default_args,
    tags=["T1", "seoul"],
) as dag:
    def _collect(**_):
        import t1_jobs, t1_steps
        return t1_jobs.collect_produce(t1_steps.slim)

    def _consume(**_):
        import t1_jobs
        return t1_jobs.consume_to_lake()

    def _spark(**_):
        import t1_jobs
        from pyspark.sql import SparkSession
        return t1_jobs.spark_congestion(SparkSession.builder.getOrCreate())

    def _rag(**_):
        import t1_jobs, t1_steps
        return t1_jobs.refresh_rag(t1_steps.docs_from)

    collect = PythonOperator(task_id="collect_produce", python_callable=_collect)
    consume = PythonOperator(task_id="consume_to_lake", python_callable=_consume)
    spark = PythonOperator(task_id="spark_congestion", python_callable=_spark)
    rag = PythonOperator(task_id="refresh_rag", python_callable=_rag)

    collect >> consume >> spark >> rag

Writing dags/t1_city_hourly.py


4단계의 `slim()` 과 7단계의 `docs_from()` 도 DAG 가 쓰도록 파일로 옮깁니다. 노트북에 정의한 함수를 `inspect.getsource` 로 꺼내 그대로 적습니다 — 두 곳에 따로 고쳐 쓰다 어긋나는 일을 막습니다.

In [ ]:
import inspect
open("dags/t1_steps.py", "w", encoding="utf-8").write(
    "from datetime import datetime\nfrom zoneinfo import ZoneInfo\nKST = ZoneInfo('Asia/Seoul')\n\n\n"
    + "\n\n".join(inspect.getsource(f) for f in (to_int, slim, docs_from)))
print(sorted(os.listdir("dags")))

['t1_city_hourly.py', 't1_jobs.py', 't1_steps.py']


Airflow 가 매시간 할 일을 노트북에서 한 번 손으로 돌려 봅니다. 수집 → 적재 → 집계 → 문서 갱신이 함수만으로 이어지는지 확인하는 것입니다.

In [ ]:
sys.path.insert(0, os.path.abspath("dags"))
import t1_jobs, t1_steps
n_sent_left = t1_jobs.collect_produce(t1_steps.slim)
n_new = t1_jobs.consume_to_lake()
n_area = t1_jobs.spark_congestion(spark)
n_docs, n_stored = t1_jobs.refresh_rag(t1_steps.docs_from, emb)
print("남은 메시지", n_sent_left, "· 새 파일", n_new, "· 장소", n_area, "· 문서", n_docs, "· Qdrant", n_stored)

남은 메시지 0 · 새 파일 1 · 장소 1 · 문서 18 · Qdrant 18


> **확인 포인트** — `남은 메시지 0 · 새 파일 1 · 장소 1`(sample) — 새로 받은 한 건만 적재됩니다. Consumer 그룹 `t1-lake` 가 5단계에서 어디까지 읽었는지 기억하기 때문입니다.
> 그리고 `문서` 와 `Qdrant` 수가 같습니다 — 장소의 옛 문서를 지우고 새로 넣으므로, 매시간 돌아도 문서가 쌓이지 않습니다.

In [ ]:
assert n_sent_left == 0 and n_new == len(PLACES) and n_area == len(PLACES)
assert n_docs == n_stored and n_docs >= 2 * len(PLACES)

이제 DAG 파일이 Airflow 에서 제대로 읽히는지 검사합니다. 노트북 venv 에는 Airflow 가 없으니, 먼저 일부러 여기서 읽어 보고 실패를 확인하세요.

In [ ]:
try:
    import t1_city_hourly
except ModuleNotFoundError as e:
    print("여기서는 못 읽음:", e)

여기서는 못 읽음: No module named 'airflow'


진짜 검사는 Airflow 2.9.3 컨테이너 안에서 합니다(8주차와 같은 판). 메타 DB 를 만들고(`db migrate`), 읽기 오류 목록과 태스크 목록을 봅니다. 처음에는 이미지(약 2GB)를 받느라 몇 분 걸릴 수 있습니다.

In [ ]:
out = subprocess.run(
    ["docker", "compose", "-p", "t1", "--profile", "airflow", "run", "--rm", "airflow", "bash", "-c",
     "airflow db migrate > /dev/null 2>&1 && airflow dags list-import-errors && airflow tasks list t1_city_hourly"],
    capture_output=True, text=True)
print(out.stdout[-600:])


No data found
collect_produce
consume_to_lake
refresh_rag
spark_congestion



> **확인 포인트** — 읽기 오류는 `No data found`, 태스크는 `collect_produce · consume_to_lake · refresh_rag · spark_congestion` 넷이 찍힙니다.

In [ ]:
tasks = [t for t in out.stdout.split() if t in ("collect_produce", "consume_to_lake", "spark_congestion", "refresh_rag")]
assert out.returncode == 0 and "No data found" in out.stdout and len(tasks) == 4

> **Aebon's Tip** — 실제로 매시간 돌리려면 8주차의 Airflow compose(웹서버 · 스케줄러)에 `dags/` 를 마운트하고, `_PIP_ADDITIONAL_REQUIREMENTS` 로 `confluent-kafka minio qdrant-client` 를 넣습니다.
> Airflow 이미지에는 Java · Spark 가 없어서, 운영에서는 Spark 태스크를 `SparkSubmitOperator` 나 `apache/spark:3.5.3` 컨테이너를 띄우는 `DockerOperator` 로 바꾸는 것이 보통입니다. 오케스트레이션 담당이 이 전환을 맡으세요.

## 10단계 · 정리와 더 해 보기 (오케스트레이션)

Spark 를 멈추고 컨테이너와 볼륨을 지웁니다. `down -v` 의 `-v` 가 MinIO · Qdrant 볼륨까지 지웁니다 — 남겨 두고 싶으면 `-v` 를 빼세요.

In [ ]:
spark.stop()
del producer, admin                                    # Kafka 클라이언트를 먼저 닫아야 브로커가 내려간 뒤 재접속을 시도하지 않습니다
!docker compose --progress quiet -p t1 --profile airflow down -v
left = subprocess.run(["docker", "ps", "-a", "-q", "--filter", "label=com.docker.compose.project=t1"], capture_output=True, text=True).stdout.split()
print("남은 컨테이너:", len(left))
assert left == []

남은 컨테이너: 0


> **확인 포인트** — `남은 컨테이너: 0`.

더 해 보기 — 역할별로 하나씩 골라 키우세요.

- 수집 · 스트리밍: 내 키로 장소를 10곳 이상으로 늘리고, Producer 를 1분마다 도는 스크립트로 떼어 내 하루 동안 쌓아 보세요. 5분 갱신 주기와 1분 수집 사이에서 중복이 얼마나 생기는지 6단계 숫자로 확인합니다.
- 레이크 · Spark: 하루치가 쌓이면 시간대(`hour`)별 평균 혼잡도와 따릉이 거치율(`bike_rate`)의 관계를 그려 보세요. `bike.live` 원본을 대여소 좌표로 장소와 이어 붙이는 것도 좋은 과제입니다.
- RAG · 서비스: T3 처럼 Streamlit 화면을 붙여 「장소 고르기 + 질문」 상황판을 만드세요(8주차). 숫자 카드는 마트 표에서, 답은 8단계의 `ask()` 에서 옵니다.
- 오케스트레이션: `refresh_rag` 를 완성하고(7단계 `docs_from` 을 `t1_jobs.py` 로 옮기기), 한 태스크가 실패했을 때 재시도 2번 뒤 멈추는지 Airflow 화면에서 확인하세요.